# Global Code

In [ ]:
import os
from huggingface_hub import login
os.environ["HF_"]

In [ ]:
from pydantic import BaseModel, SkipValidation
from functools import cache
from transformers import AutoTokenizer, AutoModelForCausalLM

# Refrain from calling these inside the notebook if cache is not available. 
# Takes too long. Use the console instead
from contextlib import redirect_stdout
import io

class ModelConfig(BaseModel):
    # This is required to set the member typehints to Auto*
    model_config = dict(arbitrary_types_allowed=True)

    model_id: str
    tokenizer: SkipValidation[AutoTokenizer]
    model: SkipValidation[AutoModelForCausalLM]
    
    # The order of decorators does matter
    @classmethod
    @cache
    def from_pretrained(cls, model_id: str):
        with redirect_stdout(io.StringIO()):
            return ModelConfig(            
                model_id=model_id,
                tokenizer=AutoTokenizer.from_pretrained(
                    pretrained_model_name_or_path=model_id, 
                    # NEVER forget this for causal models
                    padding_side="left"
                ),
                model=AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=model_id)
            )
    
    def generate(self, query: str, max_new_tokens=512, temperature=.7, top_p=.9) -> str:
        inputs = self.tokenizer(query, return_tensors="pt")
        token_ids = self.model.generate(
            **inputs, 
            # Check https://huggingface.co/docs/transformers/llm_tutorial
            do_sample=True, 
            max_new_tokens=max_new_tokens, 
            temperature=temperature, 
            top_p=top_p,
            num_beams=4
        )
        return self.tokenizer.batch_decode(
            token_ids, 
            skip_special_tokens=True, clean_up_tokenization_spaces=False
        )

# `meta-llama/Llama-3.2-1B-Instruct`

In [61]:
llm = ModelConfig.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")

In [62]:
# Call generate using a plain and an annotated prompt
plain_prompt = "Write 5 sentences of at least 5 words each, so that each sentence contains at least 1 word more than its orevious sentence."

annotated_prompt = """
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a helpful assistant.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Write 5 sentences of at least 5 words each, so that each sentence contains at least 1 word more than its orevious sentence.<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""

In [ ]:
plain_response = llm.generate(query=plain_prompt)
annotated_response = llm.generate(query=annotated_prompt)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


# `meta-llama/Llama-3.2-1B`

In [69]:
llm = ModelConfig.from_pretrained("meta-llama/Llama-3.2-1B")

In [70]:
# Call generate using a plain and an annotated prompt
plain_prompt = "Write 5 sentences of at least 5 words each, so that each sentence contains at least 1 word more than its orevious sentence."

annotated_prompt = """
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a helpful assistant.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Write 5 sentences of at least 5 words each, so that each sentence contains at least 1 word more than its orevious sentence.<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""

In [71]:
plain_response = llm.generate(query=plain_prompt)
annotated_response = llm.generate(query=annotated_prompt)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [ ]:
import os
print(annotated_response[0][:128], plain_response[0][:128], sep=os.linesep)

# ARCV

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(load_in_4bit=True)
# model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1", device_map="auto", quantization_config=quantization_config)

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float32",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "fp4",
  "bnb_4bit_use_double_quant": false,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}